# 🏮 Game Localisation Studio v0.3
Direct AI agents that localise a Hong Kong mystery game into English.

> **You are the localisation director. The agents are collaborators, not authorities.**

- 🎬 **AI Director** — configures the agents and approves the final text.
- 🔍 **Quality Evaluator** — challenges the outputs and argues for better wording.

⏱ Mission 1 ≈80 min (Step 0 ≈3 min). Missions 2–4: 35–40 min each — skip Section 1.

Click ▶ on each cell. Your work saves automatically.

In [ ]:
#@title ⚙️ Step 0 · Set up the studio — click ▶ and wait (about 20 seconds) { display-mode: "form" }
REPO_URL = "https://github.com/enricoconductive/game-localisation-studio"
BUNDLE_NAME = "studio_bundle.zip"
COLAB_TARGET = "/content/game-localisation-studio"
MAX_BUNDLE_BYTES = 40_000_000        # size of the uploaded zip
MAX_UNPACKED_BYTES = 80_000_000      # total size after unpacking
MAX_BUNDLE_FILES = 3000

import html, io, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _safe_extract(data, dest):
    """Unpack the bundle zip safely: no absolute paths, no '..', no links, size caps."""
    if len(data) > MAX_BUNDLE_BYTES:
        raise ValueError("The file is too large to be the studio bundle.")
    try:
        zf = zipfile.ZipFile(io.BytesIO(data))
    except zipfile.BadZipFile:
        raise ValueError("This is not a zip file. Upload studio_bundle.zip from your instructor.")
    infos = zf.infolist()
    if len(infos) > MAX_BUNDLE_FILES:
        raise ValueError("The zip has too many files to be the studio bundle.")
    total = 0
    dest = Path(dest).resolve()
    for info in infos:
        name = info.filename.replace("\\", "/")
        parts = PurePosixPath(name).parts
        if (name.startswith("/") or (parts and ":" in parts[0]) or ".." in parts
                or (info.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError(f"Unsafe entry in the zip ({name!r}). Ask your instructor for a new bundle.")
        total += info.file_size
        if total > MAX_UNPACKED_BYTES:
            raise ValueError("The zip unpacks to more data than the studio bundle should.")
    for info in infos:
        target = (dest / info.filename.replace("\\", "/")).resolve()
        if dest != target and dest not in target.parents:
            raise ValueError("Unsafe entry in the zip. Ask your instructor for a new bundle.")
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info) as src, open(target, "wb") as out:
            shutil.copyfileobj(src, out)
    return dest


def _find_root(base):
    base = Path(base)
    for cand in [base] + [p for p in base.iterdir() if p.is_dir()]:
        if (cand / "src" / "studio_ui.py").is_file():
            return cand
    return None


def _install_bundle(data, target):
    tmp = Path(tempfile.mkdtemp(prefix="studio_bundle_"))
    try:
        _safe_extract(data, tmp)
        root = _find_root(tmp)
        if root is None:
            raise ValueError("This zip does not contain the studio (src/studio_ui.py is missing).")
        target = Path(target)
        if target.exists():
            shutil.rmtree(target)
        shutil.move(str(root), str(target))
        return target
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def _boot(root):
    root = str(Path(root).resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    os.environ.setdefault("STUDIO_REPO_ROOT", root)
    from src.studio_ui import Studio
    return Studio.boot(repo_root=root)


class _NotReady:
    """Stands in for the studio until setup has finished."""
    def __getattr__(self, name):
        def _explain(*args, **kwargs):
            print("⏳ The studio is not set up yet. Scroll up to Step 0 and finish it "
                  "(upload studio_bundle.zip if it asks for it), then run this cell again.")
        return _explain


def _ask_for_bundle(target):
    import ipywidgets as widgets
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    upload = widgets.FileUpload(accept=".zip", multiple=False, description="Upload bundle")
    status = widgets.HTML("<b>The studio could not be downloaded automatically</b> (the course page "
                          "may be private or offline). Upload <code>studio_bundle.zip</code> from your "
                          "instructor with the button below, or drag it into the <b>Files</b> panel on the left "
                          "and run Step 0 again.")

    def _files(value):
        if isinstance(value, dict):  # ipywidgets 7
            return [(k, bytes(v.get("content", b""))) for k, v in value.items()]
        return [(v.get("name", "upload"), bytes(v.get("content", b""))) for v in (value or ())]

    def _on_upload(change):
        global studio
        files = _files(upload.value)
        if not files:
            return
        status.value = "Unpacking…"
        try:
            root = _install_bundle(files[0][1], target)
            studio = _boot(root)
            status.value = ("✅ <b>Done.</b> The studio is installed. Continue with Section 1 below.")
        except Exception as e:  # noqa: BLE001
            status.value = "❌ " + html.escape(f"{type(e).__name__}: {e}")   # never render error text as HTML

    upload.observe(_on_upload, names="value")
    display(widgets.VBox([status, upload]))


def _update_copy(target):
    """Rerunning Step 0: fetch the instructor's latest fixes; keep the old copy if that fails."""
    try:
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        # fetch + reset (not pull): always lands on the latest course version of the code;
        # student work lives outside this folder (/content/studio_state.json, /content/exports)
        subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin", "main"],
                       check=True, capture_output=True, timeout=30, env=env)
        subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                       check=True, capture_output=True, timeout=30, env=env)
        print("Updated the studio to the latest version.")
    except Exception:
        print("Using the copy already downloaded.")


def _setup():
    global studio
    studio = _NotReady()
    env_root = os.environ.get("STUDIO_REPO_ROOT")
    if env_root and (Path(env_root) / "src" / "studio_ui.py").is_file():
        studio = _boot(env_root)
        return
    if _in_colab():
        target = Path(COLAB_TARGET)
        if (target / "src" / "studio_ui.py").is_file():
            _update_copy(target)
        else:
            print("Downloading the studio…")
            try:
                subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True,
                               capture_output=True, timeout=120,
                               env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
            except Exception:
                shutil.rmtree(target, ignore_errors=True)
        dropped = target.parent / BUNDLE_NAME   # /content/studio_bundle.zip, dragged into the Files panel
        if not (target / "src" / "studio_ui.py").is_file() and dropped.is_file():
            print("Could not download from GitHub — using", BUNDLE_NAME, "from the Files panel.")
            try:
                _install_bundle(dropped.read_bytes(), target)
            except Exception as e:  # noqa: BLE001
                print("Could not use", BUNDLE_NAME, "from the Files panel:", e)
        if (target / "src" / "studio_ui.py").is_file():
            studio = _boot(target)
        else:
            _ask_for_bundle(target)
        return
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "studio_ui.py").is_file():
            studio = _boot(cand)
            return
    print("Could not find the studio files. Set STUDIO_REPO_ROOT or open this notebook inside the repository.")


if not os.environ.get("STUDIO_SETUP_DEFINE_ONLY"):
    _setup()


---
## 1 · Connect (≈5 min)
1. You start in **Replay Mode** (prepared outputs, no key).
2. Live AI: key in **Colab Secrets** → **Load my key** → **Test connection**.
3. Problems? Click **Use Replay Mode instead**.

In [ ]:
#@title ▶ Show Section 1 · Connect { display-mode: "form" }
studio.section(1)

---
## 2 · Choose your mission (≈5 min)
1. Pick **Mission 1** first, and your **English variant**.
2. Read the source strings and the context panels.
3. Click **Start this mission**.

In [ ]:
#@title ▶ Show Section 2 · Choose your mission { display-mode: "form" }
studio.section(2)

<div style="border:3px solid #d35400;border-radius:8px;padding:8px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now</b> — the partner takes the keyboard, then change <i>“Who is deciding”</i> in the status bar.</div>

---
## 3 · Configure your agents (≈15 min)
1. Pick a translator preset, then change sliders and instructions.
2. Choose what the reviewer checks.
3. Click **Show the full prompt that will be sent**.

In [ ]:
#@title ▶ Show Section 3 · Configure your agents { display-mode: "form" }
studio.section(3)

---
## 4 · Run the agents (≈20 min)
1. **1 · Baseline**, then **2 · Run translator agent**.
2. ⭐ Change the prompt in Section 3, run the translator again, compare.
3. **3 · Run reviewer agent** — findings are only suggestions.

In [ ]:
#@title ▶ Show Section 4 · Run the agents { display-mode: "form" }
studio.section(4)

<div style="border:3px solid #d35400;border-radius:8px;padding:8px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now</b> — before the Human Director step, then change <i>“Who is deciding”</i> in the status bar.</div>

---
## 5 · Human Director (≈25 min)
Use the tabs from left to right:
1. **a · Decide** on every suggestion — say why.
2. **b · Rules**, **c · Revise**, **d · Final edits**.
3. **e · Approve**, then **Download progress**.

In [ ]:
#@title ▶ Show Section 5 · Human Director { display-mode: "form" }
studio.section(5)

---
## 6 · Test and export (≈10 min)
1. Answer the reflection question; click **Save reflection**.
2. Click **▶ Play my translation in the game**.
3. Export the three files to hand in.

In [ ]:
#@title ▶ Show Section 6 · Test and export { display-mode: "form" }
studio.section(6)

<div style="border:3px solid #d35400;border-radius:8px;padding:8px 14px;background:#fff4e5;color:#1f2933"><b>🔁 Swap roles now</b> — before the next mission (back to Section 2), then change <i>“Who is deciding”</i> in the status bar.</div>

---
## 🆘 If you get stuck
- **Disconnected?** Run Step 0, then **Resume my saved session**.
- **Runtime reset?** **Resume from file** with your downloaded progress file.
- **Behind?** Pick a **Checkpoint:** and click **Load this checkpoint**.
- **Game link not working?** Export the game file; in the game use **Import student JSON**.

In [ ]:
#@title ▶ Show the help box { display-mode: "form" }
studio.stuck()